<a href="https://colab.research.google.com/github/TTS-77/Cyclone-AI/blob/main/Notebook/Batches_Creation_and_Final_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Batches Creation and Final Dataset

!pip -q install xarray zarr fsspec gcsfs pandas numpy

In [ ]:
from google.colab import files

uploaded = files.upload()

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np
import xarray as xr

track = pd.read_csv(
    "ibtracs.NI.list.v04r01.csv",
    skiprows=[1]
)

track["ISO_TIME"] = pd.to_datetime(
    track["ISO_TIME"],
    errors="coerce"
)

# 2000–2025 + North Indian Ocean model region
track = track[
    (track["SEASON"] >= 2000) &
    (track["SEASON"] <= 2025) &
    (track["LAT"] >= 0) &
    (track["LAT"] <= 30) &
    (track["LON"] >= 40) &
    (track["LON"] <= 100)
].copy()

track = track.sort_values(
    ["SID", "ISO_TIME"]
).reset_index(drop=True)

print("Filtered observations:", len(track))

/tmp/ipykernel_2856/1793032713.py:5: DtypeWarning: Columns (172,173) have mixed types. Specify dtype option on import or set low_memory=False.
  track = pd.read_csv(


Filtered observations: 8817


In [ ]:
CDS_API_KEY = input("Enter your CDS API key: ")

Enter your CDS API key: af24e5ed-22e4-4cec-9239-b882bf231ed0


In [ ]:
surface_time_url = (
    "https://arco.datastores.ecmwf.int/"
    "cadl-arco-time-002/arco/"
    "reanalysis_era5_single_levels/sfc/"
    "timeChunked.zarr"
)

ds_time = xr.open_zarr(
    surface_time_url,
    consolidated=True,
    storage_options={
        "headers": {
            "Authorization": f"Bearer {CDS_API_KEY}"
        }
    }
)

print("ERA5 connected.")

ERA5 connected.


In [ ]:
era5_lookup = track[
    [
        "SID",
        "SEASON",
        "NAME",
        "ISO_TIME",
        "LAT",
        "LON",
        "USA_WIND",
        "USA_PRES"
    ]
].copy()

era5_lookup["ERA5_TIME"] = (
    era5_lookup["ISO_TIME"].dt.floor("h")
)

In [ ]:
storm_ranges = (
    era5_lookup
    .groupby(["SID", "SEASON", "NAME"])
    .agg(
        start_time=("ERA5_TIME", "min"),
        end_time=("ERA5_TIME", "max"),
        min_lat=("LAT", "min"),
        max_lat=("LAT", "max"),
        min_lon=("LON", "min"),
        max_lon=("LON", "max"),
        observations=("SID", "size")
    )
    .reset_index()
)

print("Cyclones:", len(storm_ranges))

Cyclones: 264


In [ ]:
BATCH_SIZE = 25
BATCH_NUMBER = 11

start_idx = (BATCH_NUMBER - 1) * BATCH_SIZE
end_idx = min(
    BATCH_NUMBER * BATCH_SIZE,
    len(storm_ranges)
)

batch_storms = storm_ranges.iloc[
    start_idx:end_idx
]

print(
    f"Processing cyclones "
    f"{start_idx + 1}–{end_idx}"
)

Processing cyclones 251–264


In [ ]:
import gc
import time

ERA5_VARS = [
    "u10",
    "v10",
    "t2m",
    "d2m",
    "msl",
    "sst"
]

PADDING = 0.5

batch_results = []

start_time = time.time()

for _, storm in batch_storms.iterrows():

    sid = storm["SID"]

    # IBTrACS observations for this cyclone
    obs = era5_lookup[
        era5_lookup["SID"] == sid
    ].copy()

    if len(obs) == 0:
        continue

    # Small spatial padding
    lat_min = max(
        0,
        storm["min_lat"] - PADDING
    )

    lat_max = min(
        30,
        storm["max_lat"] + PADDING
    )

    lon_min = max(
        40,
        storm["min_lon"] - PADDING
    )

    lon_max = min(
        100,
        storm["max_lon"] + PADDING
    )

    # Get only this cyclone's ERA5 block
    era5_block = ds_time[
        ERA5_VARS
    ].sel(
        time=slice(
            storm["start_time"],
            storm["end_time"]
        ),
        latitude=slice(
            lat_min,
            lat_max
        ),
        longitude=slice(
            lon_min,
            lon_max
        )
    ).load()

    # Match IBTrACS observations to nearest ERA5 grid
    extracted = era5_block.sel(
        time=xr.DataArray(
            obs["ERA5_TIME"].values,
            dims="points"
        ),
        latitude=xr.DataArray(
            obs["LAT"].values,
            dims="points"
        ),
        longitude=xr.DataArray(
            obs["LON"].values,
            dims="points"
        ),
        method="nearest"
    )

    # Build merged dataframe
    result = obs.reset_index(drop=True).copy()

    result["ERA5_LAT"] = extracted[
        "latitude"
    ].values

    result["ERA5_LON"] = extracted[
        "longitude"
    ].values

    result["u10"] = extracted[
        "u10"
    ].values

    result["v10"] = extracted[
        "v10"
    ].values

    result["t2m"] = extracted[
        "t2m"
    ].values

    result["d2m"] = extracted[
        "d2m"
    ].values

    result["msl"] = extracted[
        "msl"
    ].values

    result["sst"] = extracted[
        "sst"
    ].values

    batch_results.append(result)

    print(
        f"{storm['NAME']} | "
        f"{len(result)} observations"
    )

    del era5_block
    del extracted
    gc.collect()

# Combine this batch
batch_df = pd.concat(
    batch_results,
    ignore_index=True
)

elapsed = time.time() - start_time

print(
    f"\nBatch {BATCH_NUMBER} complete."
)
print(
    f"Rows: {len(batch_df)}"
)
print(
    f"Time: {elapsed/60:.2f} minutes"
)

UNNAMED | 6 observations
UNNAMED | 23 observations
UNNAMED | 15 observations
UNNAMED | 7 observations
KAJIKI | 2 observations
UNNAMED | 33 observations
BUALOI | 6 observations
UNNAMED | 13 observations
UNNAMED | 17 observations
SHAKHTI | 47 observations
UNNAMED | 79 observations
MONTHA | 39 observations
SENYAR | 16 observations
DITWAH | 49 observations

Batch 11 complete.
Rows: 352
Time: 30.74 minutes


In [ ]:
batch_df["wind_speed"] = np.sqrt(
    batch_df["u10"]**2 +
    batch_df["v10"]**2
)

batch_df["t2m_C"] = (
    batch_df["t2m"] - 273.15
)

batch_df["d2m_C"] = (
    batch_df["d2m"] - 273.15
)

batch_df["sst_C"] = (
    batch_df["sst"] - 273.15
)

batch_df["msl_hPa"] = (
    batch_df["msl"] / 100
)

In [ ]:
batch_file = (
    f"IBTrACS_ERA5_batch_{BATCH_NUMBER:02d}.csv"
)

batch_df.to_csv(
    batch_file,
    index=False
)

print("Saved:", batch_file)

Saved: IBTrACS_ERA5_batch_11.csv


In [ ]:
import glob

batch_files = sorted(
    glob.glob("IBTrACS_ERA5_batch_*.csv")
)

all_data = pd.concat(
    [
        pd.read_csv(f)
        for f in batch_files
    ],
    ignore_index=True
)

all_data = all_data.sort_values(
    ["SID", "ISO_TIME"]
).reset_index(drop=True)

all_data.to_csv(
    "IBTrACS_ERA5_NI_2000_2025.csv",
    index=False
)

print("FINAL DATASET CREATED")
print("Rows:", len(all_data))
print("Cyclones:", all_data["SID"].nunique())

FINAL DATASET CREATED
Rows: 8817
Cyclones: 264
